# e-Stat 需要データ取得・結合ノートブック

国勢調査メッシュ統計や経済センサスの飲食業データを e-Stat API から取得し、既存のホットペッパーメッシュグリッドと結合して、出店候補地の需要スコアリングを改善するためのプロトタイプです。


In [ ]:
import sys
from pathlib import Path

# プロジェクトルートをパスに追加
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'IPAexGothic'
from IPython.display import display

from config.settings import ESTAT_API_KEY, ESTAT_API_BASE_URL, PROJECT_ROOT as PROJ_ROOT
from src.collect.estat import get_stats_list, get_stats_data, fetch_mesh_population, save_raw

print(f"ESTAT_API_KEY: {'設定済み' if ESTAT_API_KEY else '未設定'}")
print(f"ESTAT_API_BASE_URL: {ESTAT_API_BASE_URL}")

## Section 1: e-Stat API 動作確認


In [ ]:
# src/collect/estat.py のget_stats_listを使用
df_list = get_stats_list("国勢調査 地域メッシュ 人口")
display(df_list)

## Section 2: 国勢調査メッシュ統計（人口・昼間人口）の取得


In [ ]:
# 2020年国勢調査 地域メッシュ統計（1kmメッシュ）
# statsDataId は get_stats_list() の結果から確認・変更してください
CENSUS_MESH_ID = "0003412636"
print(f"使用する統計表ID: {CENSUS_MESH_ID}")

In [ ]:
# 東京付近（1次メッシュコード: 5339）を試験取得
SAMPLE_MESH1 = "5339"

try:
    df_mesh_pop = get_stats_data(CENSUS_MESH_ID, cd_area=SAMPLE_MESH1, limit=500)
    print(f"取得件数: {len(df_mesh_pop)}")
    display(df_mesh_pop.head(10))
    print(f"\nカラム: {df_mesh_pop.columns.tolist()}")
except Exception as e:
    print(f"取得エラー: {e}")
    # モックデータで継続
    df_mesh_pop = pd.DataFrame({
        "area": ["533900001", "533900002", "533900003", "533900004", "533900005"],
        "area_name": ["メッシュ1", "メッシュ2", "メッシュ3", "メッシュ4", "メッシュ5"],
        "value": ["12500", "8300", "21000", "5600", "18200"],
        "cat01": ["001", "001", "001", "001", "001"],
        "cat01_name": ["総人口", "総人口", "総人口", "総人口", "総人口"],
    })
    print("モックデータを使用します")
    display(df_mesh_pop)

## Section 3: 経済センサス 飲食業事業所数の取得


In [ ]:
# 経済センサス‐活動調査（飲食業）
ECENSUS_ID = "0003441172"

try:
    df_ecensus = get_stats_data(ECENSUS_ID, limit=200)
    # 飲食業関連でフィルタ
    if "cat01_name" in df_ecensus.columns:
        df_food = df_ecensus[df_ecensus["cat01_name"].str.contains("飲食", na=False)]
    elif "cat01" in df_ecensus.columns:
        df_food = df_ecensus.head(20)
    else:
        df_food = df_ecensus.head(20)
    print(f"飲食業データ件数: {len(df_food)}")
    display(df_food.head(10))
except Exception as e:
    print(f"取得エラー: {e}")
    df_food = pd.DataFrame({
        "area": ["13101", "13102", "13103", "14101", "23101"],
        "area_name": ["千代田区", "中央区", "港区", "横浜市鶴見区", "名古屋市千種区"],
        "value": ["850", "1200", "950", "430", "380"],
        "cat01_name": ["飲食店"] * 5,
    })
    print("モックデータを使用します")
    display(df_food)

## Section 4: 既存ホットペッパーデータとの結合


In [ ]:
# 既存のホットペッパーデータを読み込む
raw_dir = PROJECT_ROOT / "data" / "raw"
csv_files = list(raw_dir.glob("*_hotpepper.csv"))

if csv_files:
    df_hp = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    print(f"ホットペッパーデータ: {len(df_hp)}件")
    display(df_hp.head(3))
else:
    print("ホットペッパーデータなし → モックデータを使用")
    np.random.seed(42)
    n = 100
    lats = np.random.uniform(35.60, 35.75, n)
    lngs = np.random.uniform(139.65, 139.80, n)
    df_hp = pd.DataFrame({
        "id": [f"J{i:06d}" for i in range(n)],
        "name": [f"店舗{i}" for i in range(n)],
        "lat": lats,
        "lng": lngs,
        "genre_name": np.random.choice(["ラーメン", "カフェ", "居酒屋", "焼肉", "和食"], n),
        "mesh_code": [
            f"{int((lat - 35.60) / 0.005)}_{int((lng - 139.65) / 0.00625)}"
            for lat, lng in zip(lats, lngs)
        ],
    })

# mesh_code列がなければ生成
if "mesh_code" not in df_hp.columns and "lat" in df_hp.columns:
    LAT_STEP, LON_STEP = 0.005, 0.00625
    df_hp["mesh_code"] = (
        df_hp["lat"].apply(lambda x: f"{int(x / LAT_STEP)}") + "_" +
        df_hp["lng"].apply(lambda x: f"{int(x / LON_STEP)}")
    )

# メッシュ別集計
df_mesh_agg = (
    df_hp.groupby("mesh_code")
    .agg(
        restaurant_count=("id", "count"),
        lat_mean=("lat", "mean"),
        lng_mean=("lng", "mean"),
    )
    .reset_index()
)
print(f"メッシュ数: {len(df_mesh_agg)}")
display(df_mesh_agg.head(5))

In [ ]:
# モックの人口データをメッシュに付与して結合デモ
# 実データ取得後は下記コメントアウトを解除してmergeに差し替える
np.random.seed(0)
df_mesh_agg["population"] = np.random.randint(3000, 30000, len(df_mesh_agg))
df_mesh_agg["daytime_population"] = (
    df_mesh_agg["population"] * np.random.uniform(0.8, 3.0, len(df_mesh_agg))
)

# e-Stat実データが取得できた場合はここでmerge
# df_mesh_agg = df_mesh_agg.merge(df_mesh_pop_pivot, on="mesh_code", how="left")

display(df_mesh_agg.head(5))

## Section 5: 改善スコアリング（人口ベース需要）


In [ ]:
from src.analyze.scoring import (
    compute_opportunity_score, compute_opportunity_score_v2, _normalize
)

df_scored = df_mesh_agg.copy()

# --- 旧スコア (v1: restaurant_count ベース) ---
df_v1 = compute_opportunity_score(df_scored)

# --- 新スコア (v2: 人口ベース需要) ---
df_v2 = compute_opportunity_score_v2(df_scored)

print("=== v1 スコア上位10件（飲食店数ベース）===")
display(
    df_v1.nlargest(10, "opportunity_score")[
        ["mesh_code", "restaurant_count", "demand_score",
         "competitor_density", "opportunity_score"]
    ].round(3)
)

print("\n=== v2 スコア上位10件（人口ベース需要）===")
cols_v2 = ["mesh_code", "restaurant_count", "demand_score",
           "competitor_density", "opportunity_score"]
if "population" in df_v2.columns:
    cols_v2.insert(2, "population")
if "daytime_population" in df_v2.columns:
    cols_v2.insert(3, "daytime_population")
display(df_v2.nlargest(10, "opportunity_score")[cols_v2].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 旧スコア散布図
axes[0].scatter(
    df_scored["lng_mean"], df_scored["lat_mean"],
    c=df_scored["opportunity_score_old"], cmap="RdYlGn", s=40, alpha=0.7
)
axes[0].set_title("機会スコア（旧：飲食店数ベース）")
axes[0].set_xlabel("経度")
axes[0].set_ylabel("緯度")

# 新スコア散布図
sc = axes[1].scatter(
    df_scored["lng_mean"], df_scored["lat_mean"],
    c=df_scored["opportunity_score_new"], cmap="RdYlGn", s=40, alpha=0.7
)
axes[1].set_title("機会スコア（新：人口ベース需要）")
axes[1].set_xlabel("経度")
axes[1].set_ylabel("緯度")

plt.colorbar(sc, ax=axes[1], label="opportunity_score")
plt.tight_layout()

out_path = PROJECT_ROOT / "data" / "processed" / "opportunity_score_comparison.png"
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, dpi=100)
plt.show()
print(f"比較図を保存しました: {out_path}")

## まとめと次のステップ

- `ESTAT_API_KEY` を環境変数に設定し、モックデータ部分を実データ取得結果に差し替えて実行します。
- 国勢調査メッシュ統計の統計表IDは `get_stats_list("国勢調査 地域メッシュ 人口")` などで候補を検索して確認します。
- 経済センサス飲食業データは地域コード粒度を確認し、必要に応じて自治体コードからメッシュへの按分や補助指標として結合します。
- `scoring.py` には `daytime_population` や `competitor_density` を入力特徴量として追加し、旧来の飲食店数依存スコアを置き換える方針で組み込みます。
